In [ ]:
# Stanford Heart Transplant Data Analysis (R)
# Cox proportional hazards regression using R

# ログディレクトリの設定
logdir <- Sys.getenv("LOGDIR", "log/default")
if (!dir.exists(logdir)) {
  dir.create(logdir, recursive = TRUE)
}
cat("Log directory:", logdir, "\n")



In [ ]:
# キャッシュされたRセッションを読み込み
# build.RDataにはtargetデータフレームが含まれている
load(file.path("cache", "build.RData"))

# データの確認
cat("=== Data Summary ===\n")
cat("Dimensions:", nrow(target), "rows,", ncol(target), "columns\n")
cat("Event rate:", round(mean(target$event), 3), "\n")
cat("Unique patients:", length(unique(target$id)), "\n")
head(target)



In [ ]:
# Cox Proportional Hazards Regression
# 生存時間: stop, イベント: event
# 説明変数: transplant, age, year, surgery
# 注意: このデータセットはtime-varying covariatesを含むため、全行を使用

library(survival)
library(dplyr)
library(glue)

# ヘルパー関数の定義
format_pvalue <- function(p) {
  if (p < 0.001) return("<0.001")
  if (p < 0.01) return(sprintf("%.3f", p))
  if (p < 0.05) return(sprintf("%.3f", p))
  return(sprintf("%.3f", p))
}

format_hr <- function(x) sprintf("%.2f", x)

# モデル1: transplantのみ
cat("\n=== Model 1: Transplant only ===\n")
coxph_model1 <- coxph(Surv(stop, event) ~ transplant, data = target)
summary_model1 <- summary(coxph_model1)
print(summary_model1)

# モデル1の結果をテーブルに整形
coxph_table1 <- summary_model1$coefficients %>%
  as.data.frame() %>%
  tibble::rownames_to_column("variable") %>%
  mutate(
    HR = exp(coef),
    HR_lo = exp(coef - 1.96 * `se(coef)`),
    HR_hi = exp(coef + 1.96 * `se(coef)`),
    HR_CI = glue("{format_hr(HR)} ({format_hr(HR_lo)}-{format_hr(HR_hi)})"),
    pvalue = sapply(`Pr(>|z|)`, format_pvalue)
  ) %>%
  select(variable, HR_CI, pvalue)

print(coxph_table1)

# モデル2: transplant + age + year + surgery
cat("\n=== Model 2: Transplant + Age + Year + Surgery ===\n")
coxph_model2 <- coxph(Surv(stop, event) ~ transplant + age + year + surgery, data = target)
summary_model2 <- summary(coxph_model2)
print(summary_model2)

# モデル2の結果をテーブルに整形
coxph_table2 <- summary_model2$coefficients %>%
  as.data.frame() %>%
  tibble::rownames_to_column("variable") %>%
  mutate(
    HR = exp(coef),
    HR_lo = exp(coef - 1.96 * `se(coef)`),
    HR_hi = exp(coef + 1.96 * `se(coef)`),
    HR_CI = glue("{format_hr(HR)} ({format_hr(HR_lo)}-{format_hr(HR_hi)})"),
    pvalue = sapply(`Pr(>|z|)`, format_pvalue)
  ) %>%
  select(variable, HR_CI, pvalue)

print(coxph_table2)



In [ ]:
# Kaplan-Meier Survival Curves
# transplantでグループ分けした生存曲線を描画

library(survminer)
library(ggplot2)

# Kaplan-Meier推定
km_fit <- survfit(Surv(stop, event) ~ transplant, data = target)

# ggsurvplotを使用してプロット
km_plot <- ggsurvplot(
  km_fit,
  data = target,
  pval = TRUE,                    # Log-rank検定のp値を表示
  conf.int = TRUE,                # 信頼区間を表示
  risk.table = TRUE,              # At-risk tableを表示
  risk.table.height = 0.25,       # At-risk tableの高さ
  surv.median.line = "hv",        # 中央値生存時間の線を表示
  legend.labs = c("No Transplant", "Transplant"),  # 凡例ラベル
  xlab = "Time (days)",           # X軸ラベル
  ylab = "Survival probability",  # Y軸ラベル
  title = "Kaplan-Meier Survival Curves by Transplant Status",
  palette = c("#E7B800", "#2E9FDF"),  # カラーパレット
  ggtheme = theme_bw()            # テーマ
)

km_plot

In [ ]:

# プロットを保存
png(file.path(logdir, "kaplan_meier_curves.png"), width = 1200, height = 800, res = 150)
print(km_plot)
dev.off()

cat("\n=== Kaplan-Meier curves saved ===\n")
cat("Saved to:", file.path(logdir, "kaplan_meier_curves.png"), "\n")

# サマリー統計
cat("\n=== Kaplan-Meier Summary ===\n")
print(summary(km_fit))

# Log-rank検定
cat("\n=== Log-rank Test ===\n")
logrank_test <- survdiff(Surv(stop, event) ~ transplant, data = target)
print(logrank_test)
pval <- 1 - pchisq(logrank_test$chisq, length(logrank_test$n) - 1)
cat("Log-rank p-value:", format_pvalue(pval), "\n")



In [ ]:
# 結果をCSVとして保存
write.csv(coxph_table1, file.path(logdir, "coxph_model1.csv"), row.names = FALSE)
write.csv(coxph_table2, file.path(logdir, "coxph_model2.csv"), row.names = FALSE)

cat("\n=== Results saved ===\n")
cat("Saved to:", logdir, "\n")
cat("- coxph_model1.csv\n")
cat("- coxph_model2.csv\n")
